# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Sheheryar-khan/FlyRank-ML-Internship-Starter-Repo/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #27 — "What Predicts Health?" (Random Forest feature importance for health score)**

The paper reports Average Position (43%) and Impressions (32%) as the top predictors of `health_score`, and to its credit flags this itself: "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal." My methodology question, in that same spirit: `health_score` is defined earlier in the paper as `impressions (30pts) + position (30pts) + CTR (20pts) + scroll depth (20pts)` — so two of the top three "predictors" are literally addends of the label. This is exactly the label-derived-feature pattern from this week's leakage taxonomy (Symptom: one or two features tower over the rest). I'd ask: what does feature importance look like with `avg_position` and `impressions` excluded — does `scroll_rate` or `ctr` become informative on their own, or does importance collapse toward the base rate once the constructed pieces are removed? The paper's own caveat is the right instinct; a train-without-the-suspects run would make that caveat concrete instead of just noted.

**Finding #29 — "What Predicts Growth?" (Logistic regression, 71% holdout accuracy)**

The paper reports 71% holdout accuracy separating growing vs. declining pages, with Content Age and Days Since Update as the strongest coefficients. Two questions: First, where does the label come from — the paper defines `trend_direction` from 30-day-vs-previous-30-day impression change (Up: >10%, Down: >10%, Stable: within ±10%). That's a fine proxy, but it's worth asking whether "growing" and "declining" here means the same thing a reader might assume from the finding's title, or something narrower (a same-month momentum split, not a durable trend). Second, does the validation design carry the 71% claim — the paper doesn't say whether the holdout is a random row split or grouped by brand. With 57 brands in the portfolio, a random split risks the same issue w03's data contract flagged for this internship's own data: rows from the same brand share hidden characteristics, and a model can partly memorize brand identity rather than learn the general pattern. I'd want to see 71% reported next to a brand-grouped-holdout number, the way this week's assignment asks me to report for my own model, before treating 71% as the number that would hold up on brands the model hasn't seen.

Both questions are asked in the spirit the paper sets for itself — it already discloses its own limits well; the ask here is just to make two of those limits measurable rather than descriptive.

In [2]:
%pip -q install duckdb scikit-learn
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

features = con.sql(f"""
    SELECT content_hash_id,
        SUM(gsc_impressions) AS impressions_h1,
        SUM(gsc_clicks) AS clicks_h1,
        AVG(gsc_avg_position) AS avg_position_h1,
        COUNT(DISTINCT report_date) AS active_days_h1
    FROM {MAR}
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY content_hash_id
    HAVING impressions_h1 >= 20
""").df()
features["ctr_h1"] = features["clicks_h1"] / features["impressions_h1"] * 100

labels = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_h2
    FROM {MAR} WHERE report_date > DATE '2026-03-15' GROUP BY content_hash_id
""").df()

data = features.merge(labels, on="content_hash_id", how="left")
data["impressions_h2"] = data["impressions_h2"].fillna(0)
data["is_declining"] = (data["impressions_h2"] < 0.8 * data["impressions_h1"]).astype(int)

dim_content = con.sql(f"""
    SELECT content_hash_id, content_type, content_updated_date
    FROM read_parquet('{REL}/dim_content.parquet')
""").df()
data = data.merge(dim_content, on="content_hash_id", how="left")

client_lookup = con.sql(f"SELECT DISTINCT content_hash_id, client_hash_id FROM {MAR}").df()
data = data.merge(client_lookup, on="content_hash_id", how="left")

model_df = data.dropna(subset=["avg_position_h1", "content_type"]).copy()

numeric_features = ["impressions_h1", "clicks_h1", "avg_position_h1", "active_days_h1", "ctr_h1"]
categorical_features = ["content_type"]

def precision_at_k(scores, labels_, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels_)[order[:k]].mean()

print(model_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(109592, 11)


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

def fit_eval(train_df, test_df, label):
    X_train = train_df[numeric_features + categorical_features]
    X_test = test_df[numeric_features + categorical_features]
    y_train, y_test = train_df["is_declining"], test_df["is_declining"]

    clf = Pipeline([
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]).fit(X_train, y_train)

    proba = clf.predict_proba(X_test)[:, 1]
    return {
        "split": label,
        "precision@20": round(precision_at_k(proba, y_test.values, 20), 3),
        "precision@50": round(precision_at_k(proba, y_test.values, 50), 3),
        "ROC-AUC": round(roc_auc_score(y_test, proba), 3),
        "base_rate": round(y_test.mean(), 3),
    }

# BEFORE: naive random row split (dishonest — clients repeat across train/test)
train_rand, test_rand = train_test_split(model_df, test_size=0.25, random_state=42,
                                          stratify=model_df["is_declining"])
overlap_rand = set(train_rand["client_hash_id"]) & set(test_rand["client_hash_id"])

# AFTER: grouped split by client (honest — same as w05)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(model_df, groups=model_df["client_hash_id"]))
train_grp, test_grp = model_df.iloc[tr_idx], model_df.iloc[te_idx]
overlap_grp = set(train_grp["client_hash_id"]) & set(test_grp["client_hash_id"])

results = pd.DataFrame([
    fit_eval(train_rand, test_rand, "BEFORE — random row split"),
    fit_eval(train_grp, test_grp, "AFTER — grouped by client"),
])
print("Clients shared train/test — random split:", len(overlap_rand))
print("Clients shared train/test — grouped split:", len(overlap_grp))
results

Clients shared train/test — random split: 38
Clients shared train/test — grouped split: 0


,split,precision@20,precision@50,ROC-AUC,base_rate
0,BEFORE — random row split,0.55,0.50,0.616,0.291
1,AFTER — grouped by client,0.10,0.36,0.565,0.274


The random split overstates the model: 38 of the clients in its test set also appear in its training set, so the model had a chance to partly memorize client-specific patterns (a client's typical CTR, position range, publishing cadence) rather than learn signal that generalizes to a client it has never seen. That inflates precision@20 to 0.55 and ROC-AUC to 0.616.

Once the split is grouped by client — 0 clients shared between train and test — precision@20 drops to 0.10 (below the 0.274 base rate) and ROC-AUC drops to 0.565. This is the honest number: on a client the model has never seen, it does noticeably worse at flagging the very top of the queue, though it still shows some real lift over chance at precision@50 (0.36 vs a 0.274 base rate) and ROC-AUC (0.565 vs 0.500). The gap between 0.616 and 0.565 ROC-AUC is itself the finding — it's the size of the memorization effect the random split was hiding.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# --- 1. Timeline check: every feature is from h1 (days 1-15), label is from h2 (days 16-31) ---
print("Feature columns:", numeric_features + categorical_features)
print("Label built from: impressions_h2 (days 16-31) — strictly after all features. OK.")

# --- 2. Label-derived feature test: deliberately inject a leaky feature, confirm harness catches it ---
train_grp2 = train_grp.copy()
test_grp2 = test_grp.copy()
train_grp2["leaky_impressions_h2"] = data.loc[train_grp2.index, "impressions_h2"]
test_grp2["leaky_impressions_h2"] = data.loc[test_grp2.index, "impressions_h2"]

leaky_features = numeric_features + ["leaky_impressions_h2"]
leaky_preprocess = ColumnTransformer([
    ("num", StandardScaler(), leaky_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])
leaky_clf = Pipeline([
    ("prep", leaky_preprocess),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
]).fit(train_grp2[leaky_features + categorical_features], train_grp2["is_declining"])
leaky_auc = roc_auc_score(test_grp2["is_declining"],
                           leaky_clf.predict_proba(test_grp2[leaky_features + categorical_features])[:, 1])

honest_auc = results.loc[results["split"].str.contains("AFTER"), "ROC-AUC"].values[0]
print(f"Honest ROC-AUC (no leak): {honest_auc}")
print(f"Deliberately-leaky ROC-AUC (impressions_h2 included): {round(leaky_auc, 3)}")
print("Harness confirmed working — leaky feature jumps the score, as expected." if leaky_auc > honest_auc + 0.1
      else "Unexpected: leaky feature didn't move the score — investigate the test harness itself.")

# --- 3. Product-flag check: none of health_score/priority_score/action_type/refresh flags exist in this dataset ---
print("\\nProduct-flag columns present in data:",
      [c for c in ["health_score", "priority_score", "action_type", "refresh_tier"] if c in data.columns])

# --- 4. Full-dataset-derived feature check: was any feature calculated using stats from the WHOLE
#     dataset (including test rows) rather than train only? Yes — w04/w05's expected_ctr was.
#     Fix: recompute the position-tier CTR lookup from TRAIN ONLY, remap, and compare.
train_grp3 = train_grp.copy()
test_grp3 = test_grp.copy()

train_grp3["position_tier"] = pd.cut(train_grp3["avg_position_h1"], bins=[0,3,10,20,9999],
                                      labels=["1-3","4-10","11-20","21+"])
test_grp3["position_tier"] = pd.cut(test_grp3["avg_position_h1"], bins=[0,3,10,20,9999],
                                     labels=["1-3","4-10","11-20","21+"])

train_only_ctr = train_grp3.groupby("position_tier", observed=True).apply(
    lambda g: g["clicks_h1"].sum() / g["impressions_h1"].sum() * 100
)
full_data_ctr = data.groupby(pd.cut(data["avg_position_h1"], bins=[0,3,10,20,9999],
                                     labels=["1-3","4-10","11-20","21+"]), observed=True).apply(
    lambda g: g["clicks_h1"].sum() / g["impressions_h1"].sum() * 100
)

compare_ctr = pd.DataFrame({"train_only": train_only_ctr, "full_dataset": full_data_ctr}).round(3)
print("\\nPosition-tier expected CTR — train-only vs full-dataset (w04's original):")
print(compare_ctr)

# --- 5. Base rate printed next to every metric (already done in section 2's results table) ---
print("\\nBase rates already reported alongside every split in the section 2 table above.")

Feature columns: ['impressions_h1', 'clicks_h1', 'avg_position_h1', 'active_days_h1', 'ctr_h1', 'content_type']
Label built from: impressions_h2 (days 16-31) — strictly after all features. OK.
Honest ROC-AUC (no leak): 0.565
Deliberately-leaky ROC-AUC (impressions_h2 included): 0.995
Harness confirmed working — leaky feature jumps the score, as expected.
\nProduct-flag columns present in data: []
\nPosition-tier expected CTR — train-only vs full-dataset (w04's original):
       train_only  full_dataset
1-3         0.444         0.449
4-10        0.320         0.328
11-20       0.347         0.337
21+         0.135         0.135
\nBase rates already reported alongside every split in the section 2 table above.


/tmp/ipykernel_779/877832137.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_only_ctr = train_grp3.groupby("position_tier", observed=True).apply(


Five checks, following the attack checklist:

1. **Timeline**: all six features come from the h1 window (days 1-15); the label comes from h2 (days 16-31), strictly after. No overlap.
2. **Label-derived feature test**: deliberately adding `impressions_h2` (the column the label is thresholded from) pushed ROC-AUC from 0.565 to 0.995 — the test harness correctly catches leakage when it's present, and my honest feature set doesn't contain anything close to that.
3. **Product flags**: none of `health_score`, `priority_score`, `action_type`, or `refresh_tier` exist anywhere in this dataset — nothing to accidentally include.
4. **Full-dataset-derived stats**: my w04 baseline's `expected_ctr` was computed from the position tier's weighted CTR across the *entire* dataset, including rows that later ended up in the test split — a mild form of leakage (test-set information shaping a "training" statistic). Recomputing it from train-only rows shows the numbers barely move (1-3: 0.449→0.444, 4-10: 0.328→0.320, 11-20: 0.337→0.347, 21+: 0.135→0.135, unchanged) — at ~100K rows per tier, the aggregate is stable enough that this leakage source didn't meaningfully change the baseline, but the train-only version is the version I'd use going forward since it's the honest one by construction, not just by luck of sample size.
5. **Base rates**: printed alongside every metric in the section 2 table (0.291 for the random split's test set, 0.274 for the grouped split's) — precision numbers are read against these, not in isolation.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from w05):** "Random forest and logistic regression both clearly beat the baseline rule and the random base rate at precision@50 — logistic regression alone is picking up roughly 4x as many true declining pages in its top 50 as the hand-written rule does."

**Rewritten, safe language:** "In this single mid-panel month, on a client-grouped holdout, logistic regression's precision@50 was observed to be higher than both the baseline rule's and the random base rate's precision@50. This is a directional result from one 15-day/15-day feature/label split — it is decision-support evidence that the signals used here carry some information beyond the hand-written rule, not a proven, generalizable performance claim across months, clients, or the full warehouse."

In [ ]:
print("No code needed for this section — see markdown above.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.